# LLM Basics
In Kaggle: **Settings → Accelerator → GPU T4** and **Internet → On**

In [9]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model_name = "Qwen/Qwen2.5-0.5B-Instruct"  # small open LLM (0.5 billion parameters)

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name).to(device)
print("Loaded on", device)

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Loaded on cuda


# 1. Tokens - how an LLM sees text
Text is split into pieces (tokens) and each piece becomes a number.

In [18]:
text = "The capital of Namibia is the city of"
ids = tokenizer(text)["input_ids"]

print("Token IDs:", ids)
print("Tokens   :", [tokenizer.decode([i]) for i in ids])

Token IDs: [785, 6722, 315, 29974, 66237, 374, 279, 3283, 315]
Tokens   : ['The', ' capital', ' of', ' Nam', 'ibia', ' is', ' the', ' city', ' of']


# 2. Next word prediction - the core job of an LLM
The model gives a probability for every possible next token.

In [19]:
text = "The capital of Namibia is the city of"
inputs = tokenizer(text, return_tensors="pt").to(device)

with torch.no_grad():
    logits = model(**inputs).logits[0, -1]  # scores for the next token

probs = logits.softmax(dim=-1)
top = probs.topk(5)
for p, i in zip(top.values, top.indices):
    print(f"{tokenizer.decode([i])!r:15} {p:.1%}")

' Wind'         50.8%
' Sw'           6.5%
' Wal'          6.1%
' K'            1.9%
' Ho'           1.4%


# 3. Chat with the model
Generating an answer = predicting the next token again and again.

In [ ]:
def ask(prompt, temperature=0.7, max_new_tokens=150):

    messages = [{"role": "user", 
                "content": prompt}]

    # put the chat into the model's prompt format
    text = tokenizer.apply_chat_template(messages, 
                                         tokenize=False,              # return text, not tokens
                                         add_generation_prompt=True)  # add the assistant-turn marker

    # text -> token ids, moved to the device
    inputs = tokenizer(text, return_tensors="pt").to(device)  # pt = PyTorch tensors

    # generate the reply
    out = model.generate(**inputs, 
                         max_new_tokens=max_new_tokens,  # max reply length
                         do_sample=True,                 # random sampling, not always the top word
                         temperature=temperature)        # randomness level
    
    # tokens -> text, without the prompt
    print(tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))  # hide special tokens

In [13]:
ask("What is Linux? Answer in one sentence.")

Linux is an open-source, Unix-like operating system that was developed and maintained by Linus Torvalds for personal computers but has since become popular on servers, embedded systems, and mobile devices. It's known for its flexibility, security features, and community-driven development model, making it accessible to a wide range of users while also supporting extensive customization options. Linux provides a robust ecosystem of software tools, applications, and services, allowing users to create highly efficient and secure environments for their workloads.


# 4. Temperature - low = safe and repetitive, high = creative and random

In [21]:
for t in [0.1, 1.5]:
    print(f"--- temperature {t} ---")
    for _ in range(3):
        ask("Give me a name for a pet cat. Only the name.", temperature=t, max_new_tokens=10)

--- temperature 0.1 ---
"Whiskers"
"Whiskers"
"Whiskers"
--- temperature 1.5 ---
I'm sorry, but I cannot generate a meaningful
Casa Mia
Luna


# 5. Prompting - a better prompt gives a better answer

In [22]:
ask("Tell me about Python.")

Python is a high-level programming language that is known for its simplicity and readability. It was developed by Guido van Rossum and first released in 1991 as the successor to the small scripting language called K&R C++. Python's syntax is similar to other languages like Java, C++, and JavaScript, but it also has unique features that make it stand out.

Key aspects of Python include:

1. **Dynamic Typing**: Python allows variables to be dynamically typed, which means you can assign any type of value to a variable without knowing its data type at compile time.
2. **Syntax Flexibility**: Its syntax is relatively simple and easy to learn, making it accessible to beginners. This makes Python suitable for a wide range of applications


In [23]:
ask("You are a teacher. Explain Python to a high-school student in 3 bullet points, max 15 words each.")

Python is an interactive programming language developed by Guido van Rossum and first released in 1991. It's popular for its simplicity, readability, and wide variety of applications.

One of the primary reasons it became so widely used was due to its versatility, allowing developers to create various types of software applications quickly. Python supports multiple programming paradigms such as procedural, object-oriented, and functional.

It also has a vast ecosystem of libraries and frameworks that make it easy to build complex applications without extensive coding knowledge. This makes Python an excellent choice for beginners looking to learn how to code.


# 6. Limits - LLMs can be confidently wrong (hallucination)
The book below is made up, but the model still answers.

In [24]:
ask("Who wrote the famous novel 'The Silent River of Okahandja'? Answer in two sentences.")  # this book does not exist

"The Silent River of Okahandja" was written by the renowned American author William Faulkner. He is best known for his novels such as "A Stream of Gold" and "In the Penal Colony."
